# EDA — MSD Task01_BrainTumour (BraTS-derived)
This notebook explores:
- dataset structure (if present)
- label frequencies
- example slice overlays

If the dataset is not available, it falls back to the bundled synthetic sample under `sample/`.

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

DATA_DIR = "data/Task01_BrainTumour"
SAMPLE_DIR = "sample"

def load_nii(path):
    return nib.load(path).get_fdata()

def dataset_present():
    return os.path.exists(os.path.join(DATA_DIR, "dataset.json"))

print("Dataset present:", dataset_present())


In [ ]:
def pick_example():
    if dataset_present():
        labels_dir = os.path.join(DATA_DIR, "labelsTr")
        images_dir = os.path.join(DATA_DIR, "imagesTr")
        lbls = sorted([f for f in os.listdir(labels_dir) if f.endswith(".nii") or f.endswith(".nii.gz")])
        if not lbls:
            raise RuntimeError("labelsTr is empty.")
        label_path = os.path.join(labels_dir, lbls[0])
        case = os.path.basename(label_path).replace(".nii.gz","").replace(".nii","")
        img_paths = [os.path.join(images_dir, f"{case}_{i:04d}.nii.gz") for i in range(4)]
        img_paths = [p if os.path.exists(p) else p.replace(".nii.gz",".nii") for p in img_paths]
        return img_paths, label_path, case
    else:
        img_paths = [os.path.join(SAMPLE_DIR, f"SYNTH_{i:04d}.nii.gz") for i in range(4)]
        label_path = os.path.join(SAMPLE_DIR, "SYNTH_mask.nii.gz")
        return img_paths, label_path, "SYNTH"

img_paths, label_path, case_id = pick_example()
img_paths, label_path, case_id


In [ ]:
imgs = [load_nii(p) for p in img_paths]
mask = load_nii(label_path).astype(np.uint8)
imgs = [im[...,0] if im.ndim==4 else im for im in imgs]

print("Case:", case_id)
print("Modalities shapes:", [im.shape for im in imgs])
print("Mask shape:", mask.shape)
print("Unique labels:", np.unique(mask))


In [ ]:
# Label voxel frequencies
vals, counts = np.unique(mask, return_counts=True)
total = mask.size
for v, c in zip(vals, counts):
    print(f"Label {int(v)}: {c} voxels ({100*c/total:.3f}%)")


In [ ]:
# Overlay visualization (middle slice)
z = mask.shape[2] // 2
base = imgs[0][:, :, z]
msk = mask[:, :, z]

plt.figure(figsize=(6,6))
plt.imshow(base.T, origin="lower")
plt.imshow(np.ma.masked_where(msk.T == 0, msk.T), origin="lower", alpha=0.35)
plt.title(f"{case_id} overlay (slice z={z})")
plt.axis("off")
plt.show()


## Next EDA ideas
- per-case tumor volumes (WT/TC/ET)
- connected component counts
- intensity histograms per modality
- failure case gallery (false positives / false negatives)